In [1]:
import numpy as np

In [68]:
class GridWorld:
    def __init__(self, grid_size, n_holes=1, goal_state=None):
        self.rows, self.cols = grid_size
        
        self.goal_state = (self.rows-1, self.cols-1) if goal_state is None else goal_state   # bottom right
        self.n_holes = n_holes
        self.hole_states = self.set_holes()

        self.n_states = self.rows * self.cols
        self.action_space = [0, 1, 2, 3]  # up, down, left, right

        self.agent_pos = None

    def set_holes(self):
        n_holes = 0
        hole_states = []

        while n_holes < self.n_holes:
            while True:
                hole = (np.random.randint(0, self.rows-1), np.random.randint(0, self.cols-1))
                if hole != self.goal_state and hole not in hole_states:
                    n_holes += 1
                    hole_states.append(hole)
                    break
        return hole_states
                    

    def reset(self):
        while True:
            row = np.random.randint(0, self.rows)
            col = np.random.randint(0, self.cols)

            if (row, col) != self.goal_state:   # add any other terminal states so episode doesnt end immediately
                break
        
        self.agent_pos = (row, col)
        return self.agent_pos

    def step(self, action):

        # returns: agent pos, reward, terminated?

        row, col = self.agent_pos

        # ensure to stay within grid bounds
        if action == 0:
            row = max(row-1, 0)
        elif action == 1:
            row = min(row+1, self.rows-1)
        elif action == 2:
            col = max(col-1, 0)
        elif action == 3:
            col = min(col+1, self.cols-1)

        self.agent_pos = (row, col)

        if self.agent_pos == self.goal_state:
            return self.agent_pos, 10, True
        
        elif self.agent_pos in self.hole_states:
            return self.agent_pos, -5, False
        else:
            return self.agent_pos, -1, False



In [69]:
class MonteCarloAgent:
    def __init__(self, action_space, alpha=0.1, gamma=0.9, epsilon=0.5, epsilon_decay=0.999, min_epsilon=0.0):
        self.action_space = action_space
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        # Q(s,a)
        self.q_table = {}   # for each state, store values of each of 4 actions

    def get_q_values(self, state):

        if state not in self.q_table:
            self.q_table[state] = np.zeros(len(self.action_space))  # init each new state with 0 for each action

        return self.q_table[state]  # array of 4 actions
    
    def get_action(self, state):

        # epsilon greedy policy

        # 1. Explore
        if np.random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.action_space)
        
        # 2. Exploit
        q_values = self.get_q_values(state)
        return np.argmax(q_values)
    

    def update(self, episode):

        # FIRST VISIT MC

        states, actions, rewards = zip(*episode)

        # 1. Compute Return from rewards
        # G = R + gamma * G'

        # recursive function - unfold backwards for O(n)
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.append(G)
        
        returns.reverse()   # returns were computed from last state to first state

        # 2. First Visit update
        visited_pairs = set()    # only allow each state action pair once in 1 episode

        for (state, action, G) in zip(states, actions, returns):

            sa_pair = (state, action)

            if sa_pair not in visited_pairs:    # skip visited pairs since First Visit MC
                visited_pairs.add(sa_pair)

                q_values = self.get_q_values(state) # all q_vals for state
                old_q = q_values[action]    # Q(s,a)
                
                # Q(s,a) = Q(s,a) + alpha * (G - Q(s,a))
                new_q = old_q + self.alpha * (G - old_q)
                self.q_table[state][action] = new_q
            
        # decay eps after each episode to reduce exploration over time
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)




In [77]:
# MAIN TRAINING LOOP

env = GridWorld(grid_size=(10,10), n_holes=10, goal_state=(4,4))
agent = MonteCarloAgent(env.action_space, epsilon_decay=0.999)

episodes = 10000
incomplete_episodes = 0

for e in range(episodes):
    state = env.reset()
    done = False    # track if episode finishes (terminal state reached)
    episode_memory = [] # track (S, A, R) trajectories

    # 1. Play episode
    while not done:
        action = agent.get_action(state)
        next_state, reward, done = env.step(action)
        episode_memory.append((state, action, reward))

        state = next_state

        if len(episode_memory) > 500:    # prevent infinite loop - never reaches terminal
            incomplete_episodes += 1
            break

    # prints
    if e % 500 == 0:
        print(f"Episode: {e}   Current Epsilon: {agent.epsilon}")

        
    # 2. Learn from episode
    if done:
        agent.update(episode_memory)    # dont learn on incomplete episodes

print(f"\nIncomplete Episodes: {incomplete_episodes}")

Episode: 0   Current Epsilon: 0.5
Episode: 500   Current Epsilon: 0.30715866431907773
Episode: 1000   Current Epsilon: 0.18625454677477313
Episode: 1500   Current Epsilon: 0.11294083554888529
Episode: 2000   Current Epsilon: 0.06848494469187355
Episode: 2500   Current Epsilon: 0.04152782850113489
Episode: 3000   Current Epsilon: 0.025181600828894455
Episode: 3500   Current Epsilon: 0.015269592540540562
Episode: 4000   Current Epsilon: 0.009259159413193201
Episode: 4500   Current Epsilon: 0.005614559315273603
Episode: 5000   Current Epsilon: 0.003404550553456144
Episode: 5500   Current Epsilon: 0.002064447772331298
Episode: 6000   Current Epsilon: 0.0012518376619072773
Episode: 6500   Current Epsilon: 0.0007590880005648265
Episode: 7000   Current Epsilon: 0.0004602949808392841
Episode: 7500   Current Epsilon: 0.0002791131848062248
Episode: 8000   Current Epsilon: 0.00016924835849964345
Episode: 8500   Current Epsilon: 0.00010262864104650132
Episode: 9000   Current Epsilon: 6.22318470703

In [78]:
# Visualisation ######## goal state never reached????? in qtable

# up down inverted since (0,0) is top left
actions = [ "↑", "↓", "←", "→"]

for row in range(env.rows):
    for col in range(env.cols):

        state = (row, col)

        if state == env.goal_state:
            print("G", end=" ")
        elif state in env.hole_states:
            print("H", end=" ")
        else:
            optimal_action = np.argmax(agent.q_table[state]) # assumes every state has been visited during learning
            print(actions[optimal_action], end=" ")
    print()



→ ↓ ↓ ↓ H ↓ ↓ H ↓ ← 
→ → ↓ ↓ ↓ ← ↓ ← ← ← 
→ → → ↓ ↓ ↓ ← ← H ↑ 
H H → ↓ ↓ ← ← H H ↓ 
→ → → → G H ↓ ← ← ← 
↑ → H → ↑ ← ← ← ← ← 
→ → → → ↑ ↑ ← ↑ ← ← 
→ ↑ → ↑ ↑ H ↑ ↑ ← ← 
↑ → → → ↑ ↓ ↑ ↑ → ↑ 
↑ → ↑ ↑ ↑ ← ← ↑ ← ↑ 
